In [1]:
import sys
from pathlib import Path
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv

# 1. Resolve project root directory
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

# 2. Explicitly load .env from project root
env_path = PROJECT_ROOT / ".env"
load_dotenv(dotenv_path=env_path)

# 3. Imports and setup
import warnings
from core.state import ProjectState
from agents.agent_01_requirements import run_requirements_agent

warnings.filterwarnings("ignore", category=UserWarning)
print(f"Environment variables loaded from: {env_path}")

Environment variables loaded from: /workspaces/codespaces-blank/sdlc-multiagent-automation/.env


In [3]:
llm = init_chat_model(
    model="qwen/qwen3.6-27b",
    model_provider="groq",
    temperature=0,
    max_tokens=500  # Reduces requested output tokens below the 1000 limit
)

response = llm.invoke("Who created LangGraph?")
print(response.content)


<think>
Thinking Process:

1.  **Identify the core entity and question:** The user is asking about the creator of "LangGraph".
2.  **Retrieve knowledge about LangGraph:**
    *   What is LangGraph? It's a library for building stateful, multi-actor applications with LLMs, built on top of LangChain.
    *   Who created it? LangChain (the company) created it. Specifically, it's part of the LangChain ecosystem.
    *   Who founded LangChain? Harrison Chase.
    *   Let's double-check if there's a specific individual credited with *LangGraph* specifically, or if it's just attributed to the LangChain team/company. Usually, it's attributed to the LangChain team, led by Harrison Chase.
    *   Let's do a quick mental check or search if needed (though I should rely on internal knowledge). LangGraph was introduced by the LangChain team to handle cyclic graphs and stateful multi-agent workflows, extending LangChain's capabilities.
    *   So, the creator is the LangChain team / LangChain Inc., f

In [4]:
from core.llm_factory import get_llm

llm = get_llm(schema=None)

print("Fallbacks object:")
print(llm)

for i, fb in enumerate(llm.fallbacks):
    print(i, type(fb))


=== LLM DEBUG ===
Primary: ChatGroq
Fallback 1: ChatOpenAI
Fallback 2: ChatGoogleGenerativeAI
Fallback 3: ChatGoogleGenerativeAI
Returned Type: RunnableWithFallbacks

Fallbacks object:
runnable=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.2', 'langchain': '1.4.0'}}, profile={'name': 'Llama 3.3 70B Versatile', 'release_date': '2024-12-06', 'last_updated': '2024-12-06', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x7833c2eeb4d0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x7833c2eebed0>, model_name='llama-3.3-70b-versatile', temperature=1e-08, model_kwargs={}, groq_api_key=Secre

In [2]:
from core.llm_factory import get_llm

llm = get_llm(schema=None)

response = llm.invoke("hello")

print(response.content)


=== LLM DEBUG ===
Primary: ChatGroq
Fallback 1: ChatOpenAI
Fallback 2: ChatGoogleGenerativeAI
Fallback 3: ChatGoogleGenerativeAI
Returned Type: RunnableWithFallbacks

Hello! How can I help you today?


In [4]:
import os
from groq import Groq

client = Groq(api_key=os.getenv("GROQ_API_KEY"))

models = client.models.list()
for model in models.data:
    print(model.id)

groq/compound-mini
whisper-large-v3
canopylabs/orpheus-v1-english
allam-2-7b
groq/compound
openai/gpt-oss-120b
qwen/qwen3.6-27b
meta-llama/llama-prompt-guard-2-86m
meta-llama/llama-prompt-guard-2-22m
qwen/qwen3.8-27b
whisper-large-v3-turbo
openai/gpt-oss-safeguard-20b
canopylabs/orpheus-arabic-saudi
openai/gpt-oss-20b


In [2]:
# Pass whatever custom prompt you want to test
user_prompt = "Add aadhaar_no as VARCHAR(12) required field and alternate_phone as VARCHAR(20)"

state = ProjectState(raw_requirement=user_prompt)

In [3]:
state = run_requirements_agent(state)

# Interactively view the updated Pydantic schema contract in RAM
# Display results
print('--- AGENT 1 OUTPUT ---')
print(f'Entity: {state.current_schema.entity_name}')
print(f'Total Attributes: {len(state.current_schema.attributes)}')
print('\nNewly Added Fields:')
for attr in state.current_schema.attributes[-2:]:
    print(f' - Name: {attr.name} | Type: {attr.data_type} | Required: {attr.is_required}')


Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.



=== LLM DEBUG ===
Primary: RunnableSequence
Fallback 1: RunnableSequence
Fallback 2: RunnableSequence
Fallback 3: RunnableSequence
Fallback 4: RunnableSequence
Returned Type: RunnableWithFallbacks

--- AGENT 1 OUTPUT ---
Entity: Party
Total Attributes: 13

Newly Added Fields:
 - Name: aadhaar_no | Type: VARCHAR(12) | Required: True
 - Name: alternate_phone | Type: VARCHAR(20) | Required: False


## Agent 2 Code

In [4]:
from agents.agent_02_ui import run_ui_agent

# 1. Agent 01 updates the Pydantic schema in memory
state = run_requirements_agent(state)

# 2. Agent 02 takes the updated schema and generates Streamlit code
state = run_ui_agent(state)

print("--- AGENT 2 OUTPUT GENERATED ---")
print(f"Generated UI Code Length: {len(state.ui_code)} characters")


=== LLM DEBUG ===
Primary: RunnableSequence
Fallback 1: RunnableSequence
Fallback 2: RunnableSequence
Fallback 3: RunnableSequence
Fallback 4: RunnableSequence
Returned Type: RunnableWithFallbacks


=== LLM DEBUG ===
Primary: ChatGoogleGenerativeAI
Fallback 1: ChatGroq
Fallback 2: ChatOpenAI
Fallback 3: ChatGoogleGenerativeAI
Fallback 4: ChatGoogleGenerativeAI
Returned Type: RunnableWithFallbacks

--- AGENT 2 OUTPUT GENERATED ---
Generated UI Code Length: 9459 characters


In [16]:
# Print the exact Streamlit code Agent 2 generated
print(state.ui_code)

import streamlit as st
from datetime import date
import re

# Entity Schema provided by the user
entity_schema = {
  "domain": "Banking_Party_Master",
  "entity_name": "Party",
  "attributes": [
    {
      "name": "id_prim",
      "data_type": "VARCHAR(50)",
      "is_required": True,
      "is_primary_key": True,
      "description": "Primary Party Identifier"
    },
    {
      "name": "plss",
      "data_type": "VARCHAR(20)",
      "is_required": True,
      "is_primary_key": False,
      "description": "Prospect or Active Status"
    },
    {
      "name": "first_name",
      "data_type": "VARCHAR(100)",
      "is_required": True,
      "is_primary_key": False,
      "description": "First Name"
    },
    {
      "name": "last_name",
      "data_type": "VARCHAR(100)",
      "is_required": True,
      "is_primary_key": False,
      "description": "Last Name"
    },
    {
      "name": "tax_id_type",
      "data_type": "VARCHAR(20)",
      "is_required": False,
      "is_primary_key

## Agent 3 Code

In [5]:
from agents.agent_03_etl import run_etl_agent

# 1. Run requirements -> UI -> ETL sequential flow
state = run_requirements_agent(state)
state = run_ui_agent(state)
state = run_etl_agent(state)

print("--- AGENT 3 OUTPUT GENERATED ---")
print(f"ETL Code Length: {len(state.etl_code)} characters\n")
print(state.etl_code)


=== LLM DEBUG ===
Primary: RunnableSequence
Fallback 1: RunnableSequence
Fallback 2: RunnableSequence
Fallback 3: RunnableSequence
Fallback 4: RunnableSequence
Returned Type: RunnableWithFallbacks


=== LLM DEBUG ===
Primary: ChatGoogleGenerativeAI
Fallback 1: ChatGroq
Fallback 2: ChatOpenAI
Fallback 3: ChatGoogleGenerativeAI
Fallback 4: ChatGoogleGenerativeAI
Returned Type: RunnableWithFallbacks


=== LLM DEBUG ===
Primary: ChatGoogleGenerativeAI
Fallback 1: ChatGroq
Fallback 2: ChatOpenAI
Fallback 3: ChatGoogleGenerativeAI
Fallback 4: ChatGoogleGenerativeAI
Returned Type: RunnableWithFallbacks

--- AGENT 3 OUTPUT GENERATED ---
ETL Code Length: 9313 characters

import datetime
import re

def _clean_string(value, max_length=None):
    """Converts value to string, strips whitespace, and truncates if max_length is provided."""
    if value is None:
        return None
    s = str(value).strip()
    if not s:
        return None
    if max_length and len(s) > max_length:
        return s

## Agent 04

In [6]:
from agents.agent_04_mdm import run_mdm_agent
from pathlib import Path

# Run Agent 04 against the active state in memory
state = run_mdm_agent(state)

# Write output to schema.sql
schema_file = Path("schema.sql")
if state.mdm_ddl:
    schema_file.write_text(state.mdm_ddl)
    print("Successfully generated schema.sql!\n")
    print("--- GENERATED SQL DDL ---")
    print(state.mdm_ddl)
else:
    print("Errors occurred:", state.errors)


=== LLM DEBUG ===
Primary: ChatGoogleGenerativeAI
Fallback 1: ChatGroq
Fallback 2: ChatOpenAI
Fallback 3: ChatGoogleGenerativeAI
Fallback 4: ChatGoogleGenerativeAI
Returned Type: RunnableWithFallbacks

Successfully generated schema.sql!

--- GENERATED SQL DDL ---
-- Drop table if it already exists to ensure a clean creation (idempotency)
DROP TABLE IF EXISTS Party CASCADE;

-- Create the Party Master Data table
CREATE TABLE Party (
    -- Primary Key for the Party entity
    id_prim             VARCHAR(50)     PRIMARY KEY, -- Primary Party Identifier
    
    -- Core Party Attributes
    plss                VARCHAR(20)     NOT NULL,    -- Prospect or Active Status
    first_name          VARCHAR(100)    NOT NULL,    -- First Name
    last_name           VARCHAR(100)    NOT NULL,    -- Last Name
    tax_id_type         VARCHAR(20),                 -- Tax Identification Type
    tax_id              VARCHAR(50),                 -- Tax Identification Number
    dob                 DATE,  